In [1]:
from pathlib import Path
import pandas as pd
import re
import os
import nltk
from nltk.tokenize import word_tokenize

# Comprobar los recursos de NLTK sin descargarlos en cada ejecución
# Se usa una carpeta de usuario explícita para evitar rutas internas del entorno.
if os.name == 'nt':
    nltk_data_dir = Path(os.environ.get('APPDATA', Path.home() / 'AppData' / 'Roaming')) / 'nltk_data'
else:
    nltk_data_dir = Path.home() / 'nltk_data'
nltk_data_dir.mkdir(parents=True, exist_ok=True)
nltk.data.path = [str(nltk_data_dir)]
recursos_nltk = {
    'tokenizers/punkt': 'punkt',
    'tokenizers/punkt_tab': 'punkt_tab',
    'corpora/stopwords': 'stopwords',
}
for ruta, paquete in recursos_nltk.items():
    try:
        nltk.data.find(ruta)
    except LookupError:
        if not nltk.download(paquete, download_dir=str(nltk_data_dir), quiet=True):
            raise RuntimeError(f'No se pudo obtener el recurso NLTK: {paquete}')

# Cargar el dataset desde la carpeta data del repositorio
ROOT_DIR = Path.cwd()
if not (ROOT_DIR / 'data').exists():
    ROOT_DIR = ROOT_DIR.parent
df = pd.read_csv(ROOT_DIR / 'data' / 'libros.csv')

campos_requeridos = {'titulo', 'sinopsis', 'url_libro'}
if not campos_requeridos.issubset(df.columns):
    faltantes = campos_requeridos - set(df.columns)
    raise ValueError(f'Faltan columnas requeridas: {faltantes}')
if df['titulo'].fillna('').astype(str).str.strip().eq('').any():
    raise ValueError('El corpus contiene libros sin título.')

# Verificar la carga exitosa de los 150 libros
print(f"Total de documentos cargados en el corpus: {len(df)}")
display(df.head(2))

# Configuración del preprocesamiento diferencial exigido por la cátedra
stopwords_path = Path(nltk.data.find('corpora/stopwords')) / 'spanish'
stop_words = set(stopwords_path.read_text(encoding='utf-8').splitlines())

def limpiar_texto(texto):
    if pd.isna(texto):
        return ""
    # Pasar a minúsculas
    texto = str(texto).lower()
    # Eliminar puntuación y números
    texto = re.sub(r'[^\w\s]', '', texto)
    # Tokenizar y eliminar stopwords
    tokens = word_tokenize(texto, preserve_line=True)
    tokens_limpios = [t for t in tokens if t not in stop_words]
    return " ".join(tokens_limpios)

# 1. Versión Cruda (Para SBERT y modelos de oración)
df['texto_crudo'] = df['titulo'].fillna('') + ". " + df['sinopsis'].fillna('')

# 2. Versión Limpia (Para TF-IDF y Word2Vec propio)
df['texto_limpio'] = df['texto_crudo'].apply(limpiar_texto)

print("\n--- Verificación de transformaciones ---")
print("CRUDO:", df['texto_crudo'].iloc[0][:120])
print("LIMPIO:", df['texto_limpio'].iloc[0][:120])

Total de documentos cargados en el corpus: 150


,titulo,autores,generos,serie,sinopsis,url_libro,categoria_origen,fecha_extraccion
0,Cuando callan las montañas,Verónica Cervilla,"Drama, Histórico, Novela",NaN,"Un pueblo pirenaico, dos hermanas enfrentadas ...",https://ww3.lectulandia.co/book/cuando-callan-...,Drama,2026-09-16
1,Como los pájaros aman el aire,Martín Casariego,"Drama, Novela",NaN,Fernando lleva una existencia solitaria. Huyen...,https://ww3.lectulandia.co/book/como-los-pajar...,Drama,2026-09-16



--- Verificación de transformaciones ---
CRUDO: Cuando callan las montañas. Un pueblo pirenaico, dos hermanas enfrentadas y una acusación de brujería que cambiará sus v
LIMPIO: callan montañas pueblo pirenaico dos hermanas enfrentadas acusación brujería cambiará vidas siempre 1588 mencía inés dos


In [2]:
# Ver los nombres reales de las columnas de tu dataframe para estar seguros
print("Columnas disponibles en el CSV:", df.columns.tolist())

# El texto se mantiene como título más sinopsis; no se reemplaza por metadatos.
if df['texto_crudo'].str.strip().eq('').any():
    raise ValueError('El corpus contiene documentos de texto vacíos.')

print("\n--- Texto Crudo Corregido ---")
print(df['texto_crudo'].iloc[0])
print("\n--- Texto Limpio Corregido ---")
print(df['texto_limpio'].iloc[0])

# --- ENTRENAMIENTO DE WORD2VEC ---
from gensim.models import Word2Vec

# Preparamos los documentos tokenizados en forma de lista de listas (palabras)
corpus_tokens = [doc.split() for doc in df['texto_limpio']]

# Entrenamos nuestro propio modelo Word2Vec desde cero con el corpus de Drama
# vector_size=100 (tamaño del vector), window=5 (ventana de contexto), min_count=1 (para corpus chico de 150 libros)
w2v_model = Word2Vec(sentences=corpus_tokens, vector_size=100, window=5, min_count=1, workers=4, epochs=50)

print(f"\n¡Modelo Word2Vec entrenado con éxito!")
print(f"Vocabulario total aprendido: {len(w2v_model.wv.key_to_index)} palabras únicas.")

# Probamos una búsqueda de palabras similares si existen en el corpus
if 'amor' in w2v_model.wv:
    print("\nPalabras más similares a 'amor' según nuestro modelo de Drama:")
    print(w2v_model.wv.most_similar('amor', topn=3))

Columnas disponibles en el CSV: ['titulo', 'autores', 'generos', 'serie', 'sinopsis', 'url_libro', 'categoria_origen', 'fecha_extraccion', 'texto_crudo', 'texto_limpio']

--- Texto Crudo Corregido ---
Cuando callan las montañas. Un pueblo pirenaico, dos hermanas enfrentadas y una acusación de brujería que cambiará sus vidas para siempre . 1588 . Mencía e Inés, dos hermanas campesinas, intentan salir adelante en una villa remota del Pirineo donde la pobreza ahoga y el porvenir de las mujeres lo deciden otros. Mientras la más joven sueña con un matrimonio noble que la aleje de la miseria, Mencía acepta su destino trabajando para una de las familias más ricas y desalmadas del pueblo. Pese a tener caracteres opuestos, ambas deben aprender que hay lazos que nunca se rompen, al igual que hay secretos escondidos en lo más profundo del bosque que es mejor no conocer. Sobre todo cuando un oscuro rumor empieza a agitarse en los rincones del valle. El de las brujas. ¿Podrán escapar Mencía e Inés 

In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Cargamos un modelo SBERT liviano y multilingüe optimizado para similitud semántica
print("Cargando el modelo SBERT (esto puede demorar unos segundos la primera vez)...")
sbert_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# 2. Generamos los embeddings de todo nuestro corpus de libros usando el texto crudo
# (SBERT prefiere el texto crudo porque entiende la estructura y puntuación natural)
print("Generando vectores de embeddings para los 150 libros...")
corpus_embeddings = sbert_model.encode(df['texto_crudo'].tolist(), show_progress_bar=True)

print(f"¡Embeddings generados con éxito!")
print(f"Dimensiones de la matriz de vectores: {corpus_embeddings.shape}") # Debería ser (150, 384)

# 3. Probamos una consulta de prueba en lenguaje natural contra nuestros libros
consulta_prueba = "historias de amor trágico y desamor"
vector_consulta = sbert_model.encode([consulta_prueba])

# Calculamos la similitud de coseno entre la consulta y todos los libros del dataset
similitudes = cosine_similarity(vector_consulta, corpus_embeddings)[0]

# Obtenemos los índices de los 3 libros más similares
top_3_indices = sorted(
    range(len(similitudes)),
    key=lambda indice: similitudes[indice],
    reverse=True,
)[:3]

print(f"\n--- Resultados para la consulta: '{consulta_prueba}' ---")
for i in top_3_indices:
    print(f"Puntaje: {similitudes[i]:.4f} | Autor/Género: {df['autores'].iloc[i]} ({df['generos'].iloc[i]})")

d:\NLP-TUIA\NLP_Ayala_Cerana_Colombo_Ibarbia\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando el modelo SBERT (esto puede demorar unos segundos la primera vez)...


d:\NLP-TUIA\NLP_Ayala_Cerana_Colombo_Ibarbia\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\frank\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\NLP-TUIA\NLP_Ayala_Cerana_Colombo_Ibarbia\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggin

Generando vectores de embeddings para los 150 libros...


Batches: 100%|██████████| 5/5 [00:02<00:00,  2.44it/s]

¡Embeddings generados con éxito!
Dimensiones de la matriz de vectores: (150, 384)

--- Resultados para la consulta: 'historias de amor trágico y desamor' ---
Puntaje: 0.6094 | Autor/Género: Silvia Intxaurrondo (Drama, Novela)
Puntaje: 0.6026 | Autor/Género: William Somerset Maugham (Drama, Novela)
Puntaje: 0.5953 | Autor/Género: Maria Àngels Anglada (Drama, Novela)


In [ ]:
import json

# 1. Cargamos nuestro archivo queries.json de evaluación
with open(ROOT_DIR / 'queries.json', 'r', encoding='utf-8') as f:
    queries_eval = json.load(f)

print(f"Se cargaron {len(queries_eval)} consultas para evaluar.\n")

# 2. Función rápida para evaluar Precision@3 usando SBERT
def evaluar_buscador_sbert(queries, sbert_model, corpus_embeddings, df, k=3):
    resultados_eval = []
    
    for item in queries:
        q_id = item['id']
        q_text = item['query']
        libros_esperados = item['libros_relevantes']
        
        # Generar embedding de la consulta
        vec_q = sbert_model.encode([q_text])
        sims = cosine_similarity(vec_q, corpus_embeddings)[0]
        
        # Obtener los top k índices más similares
        top_indices = sorted(
            range(len(sims)),
            key=lambda indice: sims[indice],
            reverse=True,
        )[:k]
        
        # Extraer los slugs o identificadores de los libros predichos por el modelo
        # (Usamos el slug extraído de la url como clave de comparación)
        preds = []
        for idx in top_indices:
            url = df['url_libro'].iloc[idx]
            slug = str(url).strip('/').split('/')[-1]
            preds.append(slug)
            
        # Calcular aciertos (cuántos de los predichos están en los relevantes)
        aciertos = len(set(preds).intersection(set(libros_esperados)))
        precision = aciertos / k if k > 0 else 0
        
        resultados_eval.append({
            "id": q_id,
            "query": q_text,
            "precision_at_k": precision,
            "predicciones": preds,
            "esperados": libros_esperados
        })
        
    return resultados_eval

# Ejecutamos la evaluación
eval_resultados = evaluar_buscador_sbert(queries_eval, sbert_model, corpus_embeddings, df, k=3)

# Mostramos un resumen de los resultados por consola
print("--- RESUMEN DE EVALUACIÓN (Precision@3) ---")
precisions = []
for res in eval_resultados:
    print(f"Query {res['id']}: '{res['query']}' -> Precision@3: {res['precision_at_k']:.2f}")
    precisions.append(res['precision_at_k'])

print(f"\nPromedio general de Precision@3 del modelo SBERT: {(sum(precisions) / len(precisions) if precisions else 0.0):.2f}")